<a href="https://colab.research.google.com/github/TahaNawazkhan/Artificial-intelligence-/blob/main/lab%202%20task%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import heapq
import time
from collections import deque

START = (8, 1, 3, 4, 0, 2, 7, 6, 5)
GOALS = {
    "Goal 1 (1 2 3 / 4 5 6 / 7 8 0)": (1, 2, 3, 4, 5, 6, 7, 8, 0),
    "Goal 2 (0 1 2 / 3 4 5 / 6 7 8)": (0, 1, 2, 3, 4, 5, 6, 7, 8),
}


def solvable(state, goal):
    def inv(s):
        t = [x for x in s if x != 0]
        return sum(1 for i in range(len(t)) for j in range(i + 1, len(t)) if t[i] > t[j])
    return inv(state) % 2 == inv(goal) % 2


def successors(state):
    b = state.index(0)
    r, c = divmod(b, 3)
    for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):
        nr, nc = r + dr, c + dc
        if 0 <= nr < 3 and 0 <= nc < 3:
            n = nr * 3 + nc
            s = list(state)
            s[b], s[n] = s[n], s[b]
            yield tuple(s)


def misplaced(state, goal):
    return sum(1 for a, g in zip(state, goal) if a != 0 and a != g)


def manhattan(state, goal):
    pos = {v: i for i, v in enumerate(goal)}
    total = 0
    for i, v in enumerate(state):
        if v != 0:
            total += abs(i // 3 - pos[v] // 3) + abs(i % 3 - pos[v] % 3)
    return total


def bfs(start, goal):
    t0 = time.perf_counter()
    expanded, generated = 0, 1
    if start == goal:
        return 0, expanded, generated, time.perf_counter() - t0
    frontier = deque([(start, 0)])
    seen = {start}
    while frontier:
        state, d = frontier.popleft()
        expanded += 1
        for nxt in successors(state):
            generated += 1
            if nxt in seen:
                continue
            if nxt == goal:                      # goal test on generation
                return d + 1, expanded, generated, time.perf_counter() - t0
            seen.add(nxt)
            frontier.append((nxt, d + 1))
    return None


def astar(start, goal, h):
    t0 = time.perf_counter()
    expanded, generated = 0, 1
    tie = 0
    frontier = [(h(start, goal), 0, tie, start)]
    best_g = {start: 0}
    while frontier:
        f, g, _, state = heapq.heappop(frontier)
        if g > best_g[state]:
            continue                             # stale queue entry
        if state == goal:                        # goal test on expansion
            return g, expanded, generated, time.perf_counter() - t0
        expanded += 1
        for nxt in successors(state):
            generated += 1
            ng = g + 1
            if ng < best_g.get(nxt, float("inf")):
                best_g[nxt] = ng
                tie += 1
                heapq.heappush(frontier, (ng + h(nxt, goal), ng, tie, nxt))
    return None


if __name__ == "__main__":
    print(f"Start state: {START}\n")
    for name, goal in GOALS.items():
        print(f"{name}  solvable: {solvable(START, goal)}")
        print(f"  {'Algorithm':<22}{'Length':>7}{'Expanded':>10}{'Generated':>11}{'Time (s)':>11}")
        runs = [
            ("BFS", lambda: bfs(START, goal)),
            ("A* (misplaced tiles)", lambda: astar(START, goal, misplaced)),
            ("A* (Manhattan)", lambda: astar(START, goal, manhattan)),
        ]
        for label, fn in runs:
            length, exp, gen, secs = fn()
            print(f"  {label:<22}{length:>7}{exp:>10}{gen:>11}{secs:>11.4f}")
        print()

Start state: (8, 1, 3, 4, 0, 2, 7, 6, 5)

Goal 1 (1 2 3 / 4 5 6 / 7 8 0)  solvable: True
  Algorithm              Length  Expanded  Generated   Time (s)
  BFS                        14      3633      10092     0.0177
  A* (misplaced tiles)       14       301        827     0.0032
  A* (Manhattan)             14        76        207     0.0013

Goal 2 (0 1 2 / 3 4 5 / 6 7 8)  solvable: True
  Algorithm              Length  Expanded  Generated   Time (s)
  BFS                        22     75144     203603     0.4626
  A* (misplaced tiles)       22      9964      26924     0.1420
  A* (Manhattan)             22      1371       3668     0.0243

